# 06.10_All_rds_to_h5_R

整合 Seurat RDS 转 H5 与元数据。

- 当前文件：`analysis/06_single_cell_analysis/06.10_All_rds_to_h5_R.ipynb`
- 原始来源：`Codes/06.10_R_rds_to_h5.ipynb`（旧编号仅用于溯源）。
- 运行内核：**R**。
- 导入依赖：`BiocManager`, `Matrix`, `Seurat`, `rhdf5`。
- 当前编号与流程见 `docs/workflow.md`、`docs/code_index.md`。
- 仅更新整理版导读；原分析单元格、参数和顺序保持不变。原始 cell 索引在本文件中加 1。


r_scmap

# RDS to h5

In [ ]:
# if (!requireNamespace("BiocManager", quietly = TRUE))
#   install.packages("BiocManager")
# BiocManager::install("SeuratObject")
# BiocManager::install("Seurat")

library(Seurat)

# R code for converting RDS to h5
library(rhdf5)
library(Matrix)

# 保存为 h5 矩阵
# 改进的 save_h5mat 函数
save_h5mat = function(mat, fp_h5, feature_type, genome=""){
  
  # 检查输入
  if (is.null(mat) || length(mat) == 0) {
    stop("Input matrix is NULL or empty")
  }
  
  if (is.null(colnames(mat))) {
    stop("Matrix has no column names (barcodes)")
  }
  
  if (is.null(rownames(mat))) {
    stop("Matrix has no row names (features)")
  }
  
  message(paste("Saving to:", fp_h5))
  message(paste("Matrix dimensions:", nrow(mat), "x", ncol(mat)))
  
  # 确保是 dgCMatrix 格式
  if (!inherits(mat, "dgCMatrix")) {
    message("Converting matrix to dgCMatrix format...")
    mat <- as(mat, "dgCMatrix")
  }
  
  # 创建 HDF5 文件
  if (file.exists(fp_h5)) {
    file.remove(fp_h5)
  }
  
  h5createFile(fp_h5)
  root = "matrix"
  h5createGroup(fp_h5, root)
  
  # 写入数据
  h5write(dim(mat), fp_h5, paste(root, "shape", sep='/'))
  h5write(mat@x, fp_h5, paste(root, "data", sep='/'))
  h5write(mat@i, fp_h5, paste(root, "indices", sep='/'))
  h5write(mat@p, fp_h5, paste(root, "indptr", sep='/'))
  h5write(colnames(mat), fp_h5, paste(root, "barcodes", sep='/'))
  
  # 写入 features 信息
  feat_root = paste(root, "features", sep='/')
  h5createGroup(fp_h5, feat_root)
  
  h5write(rownames(mat), fp_h5, paste(feat_root, "id", sep='/'))
  h5write(rownames(mat), fp_h5, paste(feat_root, "name", sep='/'))
  h5write(rep(feature_type, nrow(mat)), fp_h5, paste(feat_root, "feature_type", sep='/'))
  h5write(rep("", nrow(mat)), fp_h5, paste(feat_root, "derivation", sep='/'))
  h5write(rep(genome, nrow(mat)), fp_h5, paste(feat_root, "genome", sep='/'))
  h5write(c("genome", "derivation"), fp_h5, paste(feat_root, "_all_tag_keys", sep='/'))
  
  h5closeAll()
  message("Done!")
}
# save_h5mat_peak = function(mat, fp_h5, genome=""){
#   save_h5mat(mat, fp_h5, feature_type = "Peaks", genome = genome)
# }

save_h5mat_gex = function(mat, fp_h5, genome=""){
  save_h5mat(mat, fp_h5, feature_type = "Gene Expression", genome = genome)
}

## 整合后的9 Species

In [ ]:
## save the raw-counts in a Seurat-object "seurat_obj"
# 加载 RDS 数据
SCT_UMI_expression_matrix <- readRDS("/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/SingleCellIntegrated/Seurat_RPCA/integrated_rpca.rds")
seurat_object <- SCT_UMI_expression_matrix

In [ ]:
# 查看Seurat对象的基本信息
print(seurat_object)

In [ ]:
# 查看包含的 assays
Assays(seurat_object)

In [ ]:
# 查看元数据
head(seurat_object@meta.data)

In [ ]:
# ... [your existing code for save_h5mat_gex and loading seurat_object] ...

# --- Save Dimensional Reductions ---

# Check which reductions are available
print(Reductions(seurat_object)) # Should list 'pca', 'umap', 'tsne'

# Define base path for reductions
reduction_base_path <- "/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/SingleCellIntegrated/Seurat_RPCA_to_Scanpy/sc_BasalMetazoa.reduction."

# Save PCA coordinates
if ("pca" %in% Reductions(seurat_object)) {
  pca_coords <- Embeddings(seurat_object, reduction = "pca")
  write.csv(pca_coords, paste0(reduction_base_path, "pca.csv"), row.names = TRUE)
  message("PCA embeddings saved.")
}

# Save UMAP coordinates
if ("umap" %in% Reductions(seurat_object)) {
  umap_coords <- Embeddings(seurat_object, reduction = "umap")
  write.csv(umap_coords, paste0(reduction_base_path, "umap.csv"), row.names = TRUE)
  message("UMAP embeddings saved.")
}

# Save t-SNE coordinates
if ("tsne" %in% Reductions(seurat_object)) {
  tsne_coords <- Embeddings(seurat_object, reduction = "tsne")
  write.csv(tsne_coords, paste0(reduction_base_path, "tsne.csv"), row.names = TRUE)
  message("t-SNE embeddings saved.")
}

In [ ]:
# 定义输出文件路径模板
h5_path <- "/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/SingleCellIntegrated/Seurat_RPCA_to_Scanpy/sc_BasalMetazoa.matrix.raw.h5"
csv_path <- "/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/SingleCellIntegrated/Seurat_RPCA_to_Scanpy/sc_BasalMetazoa.metadata.csv"


# 提取计数矩阵, 对于 Assay5 类型，你需要使用 GetAssayData() 函数来提取原始计数矩阵：
# mat <- GetAssayData(seurat_object, slot = "counts")
# mat <- GetAssayData(seurat_object, layer = "counts")
# save_h5mat_gex(mat, h5_path, genome="")

# save the meta-data into a csv file:
meta_data = seurat_object@meta.data
write.csv(meta_data, csv_path)

In [ ]:
# 检查矩阵的基本信息
print("=== 检查矩阵信息 ===")

# 查看可用的 assays
print("Available assays:")
print(Assays(seurat_object))

# 尝试不同的方式提取数据
print("=== 尝试提取 counts 数据 ===")

# 方法1：使用 layer 参数
mat_counts <- GetAssayData(seurat_object, layer = "counts")
print("Method 1 - counts layer:")
print(paste("Class:", class(mat_counts)))
print(paste("Dimensions:", paste(dim(mat_counts), collapse=" x ")))
print(paste("Has colnames:", !is.null(colnames(mat_counts))))
if(!is.null(colnames(mat_counts))) print(head(colnames(mat_counts), 3))

# 方法2：使用 data 参数（可能是原始数据存储在其他层）
mat_data <- GetAssayData(seurat_object, layer = "data")
print("Method 2 - data layer:")
print(paste("Class:", class(mat_data)))
print(paste("Dimensions:", paste(dim(mat_data), collapse=" x ")))
print(paste("Has colnames:", !is.null(colnames(mat_data))))

# 方法3：直接访问对象的其他可能的数据位置
if ("RNA" %in% Assays(seurat_object)) {
  print("=== 检查 RNA assay ===")
  rna_assay <- GetAssay(seurat_object, assay = "RNA")
  print(paste("RNA assay class:", class(rna_assay)))
  
  # 对于 Assay5 对象
  if (inherits(rna_assay, "Assay5")) {
    print("Layers in RNA assay:")
    print(Layers(rna_assay))
    
    # 尝试获取 count 层
    if ("counts" %in% Layers(rna_assay)) {
      mat_rna_counts <- LayerData(rna_assay, layer = "counts")
      print("RNA counts layer:")
      print(paste("Dimensions:", paste(dim(mat_rna_counts), collapse=" x ")))
    }
  }
}

# 检查 meta.data 中的细胞名称
print("=== 检查 meta.data 细胞名称 ===")
print(paste("Number of cells in meta.data:", nrow(seurat_object@meta.data)))
print(head(rownames(seurat_object@meta.data), 5))

# 修复问题的代码
print("=== 尝试修复 ===")

# 如果 mat_counts 有数据但没有列名
if (!is.null(mat_counts) && length(mat_counts) > 0) {
  if (is.null(colnames(mat_counts))) {
    print("Adding column names from meta.data")
    colnames(mat_counts) <- rownames(seurat_object@meta.data)
  }
  
  # 检查行名（基因名）
  if (is.null(rownames(mat_counts))) {
    print("Adding row names")
    # 从 features 获取基因名
    features <- rownames(GetAssayData(seurat_object, layer = "counts"))
    if (!is.null(features)) {
      rownames(mat_counts) <- features
    } else {
      rownames(mat_counts) <- paste0("gene_", 1:nrow(mat_counts))
    }
  }
  
  # 现在尝试保存
  print("Saving matrix with fixed names...")
  save_h5mat_gex(mat_counts, h5_path, genome="")
} else {
  print("mat_counts is NULL or empty")
  
  # 尝试其他可能的层
  possible_layers <- c("counts", "data", "scale.data")
  for (layer_name in possible_layers) {
    tryCatch({
      mat_alt <- GetAssayData(seurat_object, layer = layer_name)
      if (!is.null(mat_alt) && length(mat_alt) > 0) {
        print(paste("Found data in layer:", layer_name))
        
        # 添加列名和行名
        colnames(mat_alt) <- rownames(seurat_object@meta.data)
        if (is.null(rownames(mat_alt))) {
          features <- rownames(GetAssayData(seurat_object, layer = layer_name))
          if (!is.null(features)) {
            rownames(mat_alt) <- features
          }
        }
        
        # 保存
        save_h5mat_gex(mat_alt, 
                      gsub(".raw.h5", paste0(".", layer_name, ".h5"), h5_path), 
                      genome="")
        break
      }
    }, error = function(e) {
      print(paste("Error with layer", layer_name, ":", e$message))
    })
  }
}